In [3]:
#DOM hierarchy

In [4]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path
from urllib.parse import urlparse
from collections import Counter
from bs4 import BeautifulSoup

# =========================
# 1. PATHS
# =========================
JSON_PATH = Path("/Users/timvandegevel/Downloads/profiling-data/results_expanded.json")
HTML_DIR  = Path("/Users/timvandegevel/Downloads/profiling-data/html_front")
CSS_DIR   = Path("/Users/timvandegevel/Downloads/profiling-data/css_front")


# =========================
# 2. LOAD DATA
# =========================
df = pd.read_json(JSON_PATH, lines=True)

# Keep only the two thesis classes
df = df[df["mbfc-credibility-rating"].isin(["high credibility", "medium credibility"])].copy()

# Binary target
df["target"] = df["mbfc-credibility-rating"].map({
    "high credibility": 0,
    "medium credibility": 1
})


# =========================
# 3. MATCH URLS TO FILES
# =========================
html_files = set(os.listdir(HTML_DIR))
css_files = set(os.listdir(CSS_DIR))

def get_filename_from_url(url, extension):
    try:
        domain = urlparse(url).netloc
        return f"{domain}.{extension}"
    except:
        return None

df["html_filename"] = df["url"].apply(lambda x: get_filename_from_url(x, "html"))
df["css_filename"]  = df["url"].apply(lambda x: get_filename_from_url(x, "css"))

df["has_html"] = df["html_filename"].apply(lambda x: x in html_files)
df["has_css"]  = df["css_filename"].apply(lambda x: x in css_files)

# Final thesis sample: keep only rows with BOTH files
df["has_both_files"] = df["has_html"] & df["has_css"]
analysis_df = df[df["has_both_files"]].copy()

# Full paths
analysis_df["html_file_path"] = analysis_df["html_filename"].apply(lambda x: str(HTML_DIR / x))
analysis_df["css_file_path"]  = analysis_df["css_filename"].apply(lambda x: str(CSS_DIR / x))

print("Final sample shape:", analysis_df.shape)
print(analysis_df["mbfc-credibility-rating"].value_counts())

# Optional hard check
assert analysis_df.shape[0] == 3130, f"Expected 3130 rows, got {analysis_df.shape[0]}"
assert analysis_df["target"].value_counts().to_dict() == {0: 2561, 1: 569}, \
    f"Unexpected class counts: {analysis_df['target'].value_counts().to_dict()}"

Final sample shape: (3130, 44)
mbfc-credibility-rating
high credibility      2561
medium credibility     569
Name: count, dtype: int64


In [7]:
from bs4 import BeautifulSoup, Tag, Comment
from urllib.parse import urlparse, urljoin
import re

# --------------------------------------------------
# 1. BASIC HELPERS
# --------------------------------------------------
def get_tag_children(node):
    return [child for child in node.children if isinstance(child, Tag)]

def get_max_depth(node):
    if node is None:
        return 0

    max_depth = 0
    stack = [(node, 0)]

    while stack:
        current, depth = stack.pop()
        children = get_tag_children(current)

        if not children:
            if depth > max_depth:
                max_depth = depth
        else:
            for child in children:
                stack.append((child, depth + 1))

    return max_depth

def get_avg_branching(soup):
    branching = []
    for tag in soup.find_all(True):
        children = get_tag_children(tag)
        if len(children) > 0:
            branching.append(len(children))
    return sum(branching) / len(branching) if branching else 0.0


# --------------------------------------------------
# 2. NAV LIST DEPTH
# --------------------------------------------------
def get_nested_list_children(list_tag):
    nested_lists = []
    for li in list_tag.find_all("li", recursive=False):
        nested_lists.extend(li.find_all(["ul", "ol"], recursive=False))
    return nested_lists

def get_list_depth(tag, depth=1):
    children = get_nested_list_children(tag)
    if not children:
        return depth
    return max(get_list_depth(child, depth + 1) for child in children)

def get_nav_max_list_depth(soup):
    nav_tags = soup.find_all("nav")
    max_depth = 0

    for nav in nav_tags:
        lists = nav.find_all(["ul", "ol"])
        for lst in lists:
            if lst.find_parent(["ul", "ol"]) is None:
                depth = get_list_depth(lst, 1)
                if depth > max_depth:
                    max_depth = depth

    return max_depth


# --------------------------------------------------
# 3. VISIBLE TEXT
# --------------------------------------------------
TEXT_EXCLUDE_PARENTS = {"style", "script", "head", "title", "meta", "noscript", "[document]"}

def get_visible_text_length(soup):
    visible_chunks = []

    for text_node in soup.find_all(string=True):
        if isinstance(text_node, Comment):
            continue

        parent_name = text_node.parent.name if text_node.parent else None
        if parent_name in TEXT_EXCLUDE_PARENTS:
            continue

        cleaned = re.sub(r"\s+", " ", str(text_node)).strip()
        if cleaned:
            visible_chunks.append(cleaned)

    return len(" ".join(visible_chunks))


# --------------------------------------------------
# 4. URL / LINK HELPERS
# --------------------------------------------------
def normalize_netloc(url):
    try:
        netloc = urlparse(url).netloc.lower().strip()
        if netloc.startswith("www."):
            netloc = netloc[4:]
        return netloc
    except:
        return ""

PSEUDO_LINK_SCHEMES = {"javascript", "mailto", "tel", "sms", "whatsapp", "data"}

def is_internal_link(href, page_url):
    if not href:
        return False

    href = str(href).strip()
    if not href:
        return False

    # Fragment links and root-relative links count as internal
    if href.startswith("#") or href.startswith("/"):
        return True

    try:
        resolved = urljoin(page_url, href)
        parsed = urlparse(resolved)
        scheme = parsed.scheme.lower()

        # Ignore javascript/mailto/tel/etc.
        if scheme in PSEUDO_LINK_SCHEMES:
            return False

        page_domain = normalize_netloc(page_url)
        link_domain = normalize_netloc(resolved)

        if not link_domain:
            return False

        return link_domain == page_domain
    except:
        return False


def is_external_link(href, page_url):
    if not href:
        return False

    href = str(href).strip()
    if not href:
        return False

    if href.startswith("#"):
        return False

    try:
        resolved = urljoin(page_url, href)
        scheme = urlparse(resolved).scheme.lower()

        # Ignore javascript/mailto/tel/etc.
        if scheme in PSEUDO_LINK_SCHEMES:
            return False

        page_domain = normalize_netloc(page_url)
        link_domain = normalize_netloc(resolved)

        return bool(link_domain) and link_domain != page_domain
    except:
        return False


# --------------------------------------------------
# 5. NEW HTML FEATURE HELPERS
# --------------------------------------------------
def get_link_features(soup, page_url):
    a_tags = soup.find_all("a")
    total_links = len(a_tags)

    if total_links == 0:
        return {
            "internal_link_ratio": 0.0,
            "external_link_ratio": 0.0,
            "internal_to_external_link_ratio": 0.0,
            "nav_link_ratio": 0.0,
            "empty_link_ratio": 0.0
        }

    internal_links = 0
    external_links = 0
    nav_links = 0
    empty_links = 0

    for a in a_tags:
        href = a.get("href", "")

        if is_internal_link(href, page_url):
            internal_links += 1

        if is_external_link(href, page_url):
            external_links += 1

        if a.find_parent("nav") is not None:
            nav_links += 1

        visible_text = a.get_text(" ", strip=True)
        aria_label = str(a.get("aria-label", "")).strip()
        title_attr = str(a.get("title", "")).strip()

        if not visible_text and not aria_label and not title_attr:
            empty_links += 1

    # Smoothed ratio to avoid raw-count fallback / divide-by-zero issues
    internal_to_external_ratio = (internal_links + 1) / (external_links + 1)

    return {
        "internal_link_ratio": internal_links / total_links,
        "external_link_ratio": external_links / total_links,
        "internal_to_external_link_ratio": internal_to_external_ratio,
        "nav_link_ratio": nav_links / total_links,
        "empty_link_ratio": empty_links / total_links
    }

def get_heading_count(soup):
    return len(soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"]))

def get_paragraph_count(soup):
    return len(soup.find_all("p"))

def get_meta_description_present(soup):
    tag = soup.find("meta", attrs={"name": re.compile(r"^description$", re.I)})
    return 1 if tag and tag.get("content") else 0


# --------------------------------------------------
# 6. MAIN FUNCTION: DOM FEATURES
# --------------------------------------------------
def extract_dom_features(html_file_path, page_url):
    try:
        with open(html_file_path, "r", encoding="utf-8", errors="ignore") as f:
            html_content = f.read()
            soup = BeautifulSoup(html_content, "html.parser")

        all_tags = soup.find_all(True)
        total_nodes = len(all_tags)

        if total_nodes == 0:
            # Return zeros for all keys to keep the DataFrame consistent
            return {
                "dom_max_depth": 0, "dom_avg_branching": 0.0, "dom_branching_std": 0.0,
                "json_ld_present": 0, "social_link_ratio": 0.0, "semantic_tag_ratio": 0.0,
                "nav_max_list_depth": 0, "text_to_tag_ratio": 0.0, "interactive_ratio": 0.0,
                "miss_semantic_tags": 1, "internal_link_ratio": 0.0, "external_link_ratio": 0.0,
                "internal_to_external_link_ratio": 0.0, "nav_link_ratio": 0.0,
                "empty_link_ratio": 0.0, "heading_density": 0.0, "paragraph_density": 0.0,
                "meta_description_present": 0
            }

        # --- CORE CALCULATIONS ---
        root = soup.html if soup.html else soup
        dom_max_depth = get_max_depth(root)
        dom_avg_branching = get_avg_branching(soup)
        
        # 1. New Structural Consistency Feature
        branching_factors = [len(get_tag_children(tag)) for tag in all_tags]
        dom_branching_std = np.std(branching_factors) if branching_factors else 0.0

        # 2. Semantic Tag Ratio (FIX: This defines the missing variable)
        semantic_tags = ["article", "main", "nav", "section", "aside", "header", "footer"]
        semantic_count = sum(len(soup.find_all(tag)) for tag in semantic_tags)
        semantic_tag_ratio = semantic_count / total_nodes
        miss_semantic_tags = 1 if semantic_count == 0 else 0

        # 3. New Metadata & Social Features
        json_ld_present = 1 if soup.find('script', type='application/ld+json') else 0
        links = [a.get('href', '') for a in soup.find_all('a')]
        social_ks = ['facebook.com', 'twitter.com', 'linkedin.com', 'instagram.com', 'youtube.com']
        social_count = sum(1 for l in links if any(sk in str(l).lower() for sk in social_ks))
        social_link_ratio = social_count / len(links) if links else 0.0

        # 4. Existing Ratio Calculations
        nav_max_list_depth = get_nav_max_list_depth(soup)
        text_length = get_visible_text_length(soup)
        text_to_tag_ratio = text_length / total_nodes
        
        interactive_tags = ["a", "button"]
        interactive_count = sum(len(soup.find_all(tag)) for tag in interactive_tags)
        interactive_ratio = interactive_count / total_nodes
        
        link_feats = get_link_features(soup, page_url)
        
        # 5. Density Features
        heading_count = get_heading_count(soup)
        paragraph_count = get_paragraph_count(soup)
        heading_density = heading_count / total_nodes
        paragraph_density = paragraph_count / total_nodes

        meta_description_present = get_meta_description_present(soup)

        return {
            "dom_max_depth": dom_max_depth,
            "dom_avg_branching": dom_avg_branching,
            "dom_branching_std": dom_branching_std,
            "json_ld_present": json_ld_present,
            "social_link_ratio": social_link_ratio,
            "semantic_tag_ratio": semantic_tag_ratio,
            "nav_max_list_depth": nav_max_list_depth,
            "text_to_tag_ratio": text_to_tag_ratio,
            "interactive_ratio": interactive_ratio,
            "miss_semantic_tags": miss_semantic_tags,
            "internal_link_ratio": link_feats["internal_link_ratio"],
            "external_link_ratio": link_feats["external_link_ratio"],
            "internal_to_external_link_ratio": link_feats["internal_to_external_link_ratio"],
            "nav_link_ratio": link_feats["nav_link_ratio"],
            "empty_link_ratio": link_feats["empty_link_ratio"],
            "heading_density": heading_density,
            "paragraph_density": paragraph_density,
            "meta_description_present": meta_description_present
        }

    except Exception as e:
        print(f"Error processing {html_file_path}: {e}")
        # Ensure the error return has the same keys
        return {k: None for k in [
            "dom_max_depth", "dom_avg_branching", "dom_branching_std", "json_ld_present", 
            "social_link_ratio", "semantic_tag_ratio", "nav_max_list_depth", 
            "text_to_tag_ratio", "interactive_ratio", "miss_semantic_tags", 
            "internal_link_ratio", "external_link_ratio", "internal_to_external_link_ratio", 
            "nav_link_ratio", "empty_link_ratio", "heading_density", "paragraph_density", 
            "meta_description_present"
        ]}

In [8]:
# --------------------------------------------------
# 6. APPLY TO FINAL 3130-ROW SAMPLE
# --------------------------------------------------
dom_feature_rows = []

for _, row in analysis_df.iterrows():
    features = extract_dom_features(
        html_file_path=row["html_file_path"],
        page_url=row["url"]
    )
    dom_feature_rows.append(features)

dom_feature_df = pd.DataFrame(dom_feature_rows)

dom_result_df = pd.concat([analysis_df.reset_index(drop=True), dom_feature_df], axis=1)

In [13]:
import numpy as np

# 1. Drop redundant features
# We keep density and ratios, removing raw counts and binary flags
cols_to_drop = ["heading_count", "paragraph_count", "miss_semantic_tags", "internal_link_ratio"]
dom_result_df = dom_result_df.drop(columns=[c for c in cols_to_drop if c in dom_result_df.columns])

# 2. Log Transform for 'internal_to_external_link_ratio'
# This handles the range of 0 to 317, making it more linear for the model
dom_result_df["internal_to_external_link_ratio"] = np.log1p(dom_result_df["internal_to_external_link_ratio"])

# 3. Cap Outliers for 'dom_branching_std' 
# We cap at the 95th percentile to prevent extreme values (like 40.0) from biasing the model
upper_limit = dom_result_df["dom_branching_std"].quantile(0.95)
dom_result_df["dom_branching_std"] = dom_result_df["dom_branching_std"].clip(upper=upper_limit)

print(f"Transformation complete. 'dom_branching_std' capped at {upper_limit:.2f}")

Transformation complete. 'dom_branching_std' capped at 6.76


In [14]:
print(dom_result_df.groupby('target').mean(numeric_only=True))

        has_html  has_css  has_both_files  dom_max_depth  dom_avg_branching  \
target                                                                        
0            1.0      1.0             1.0      17.577509           2.011235   
1            1.0      1.0             1.0      17.214411           2.303958   

        dom_branching_std  json_ld_present  social_link_ratio  \
target                                                          
0                2.654828         0.776650           0.037475   
1                3.360914         0.667838           0.035739   

        semantic_tag_ratio  nav_max_list_depth  text_to_tag_ratio  \
target                                                              
0                 0.027284            1.235064           7.707112   
1                 0.019618            0.722320          11.314165   

        interactive_ratio  external_link_ratio  \
target                                           
0                0.177250             0.17698

In [15]:
# =========================
# SAVE OUTPUT (Updated)
# =========================
dom_cols = [
    "dom_max_depth",
    "dom_avg_branching",
    "dom_branching_std",      # Now capped
    "json_ld_present",
    "social_link_ratio",
    "semantic_tag_ratio",
    "nav_max_list_depth",
    "text_to_tag_ratio",
    "interactive_ratio",
    "external_link_ratio",
    "internal_to_external_link_ratio", # Now log-transformed
    "nav_link_ratio",
    "empty_link_ratio",
    "heading_density",
    "paragraph_density",
    "meta_description_present"
]

output_cols = [
    "url",
    "mbfc-credibility-rating",
    "target",
    "html_filename",
    "css_filename",
    "country",
    "media-type"
] + dom_cols

final_dom_df = dom_result_df[output_cols].copy()
final_dom_df.to_csv("dom_hierarchy_features_v3.csv", index=False)

print("Saved: dom_hierarchy_features_v3.csv")

Saved: dom_hierarchy_features_v3.csv
